# Astroshade — User Journey (Step by Step)

Validating each API call in the 7-step salonist flow, one cell at a time.

### The 7 steps (from Matt)
1. Upload "desired state" image + client notes
2. Confirm or correct AI description of desired state *(vision inference)*
3. Upload photo of client's starting state + consent
4. Confirm or correct AI description of starting state *(vision inference)*
5. Show preview to client *(image gen — stretch)*
6. Show formulation recommendation + colour theory, gather rating & feedback
7. CTA — "interested in product updates?" + email capture

We test each inference step against our 3 known cases before wiring anything together.

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from PIL import Image
from IPython.display import display
import json, os, textwrap

load_dotenv()
client = genai.Client()
MODEL = 'gemini-2.5-flash'
print(f'Client ready — model: {MODEL}')

---
## Steps 1–2: Desired State Analysis

**Step 1** — Client uploads a "desired look" reference image and describes what they want in their own words.

**Step 2** — AI analyses the image and returns a structured description of the desired colour. The salonist confirms or corrects before proceeding.

### What we're testing
Can Gemini look at a reference photo + client notes and return an accurate, structured colour description that a colourist would agree with?

In [ ]:
DESIRED_STATE_PROMPT = """\
You are an expert hair colour analyst. A client wants to achieve a specific hair colour result.
They may provide a reference photo, a text description, or both.

Analyse whatever the client has provided and return a structured assessment of the desired colour.
Be precise — a professional colourist will use your description to plan a formulation.

If only text is provided, infer as much as you can from the description and flag any ambiguity.
If only an image is provided, describe what you see.
If both are provided, use the image as primary and the text to clarify intent.
"""

class DesiredStateAnalysis(BaseModel):
    target_level: int = Field(description="Target colour level (1=black, 10=lightest blonde)")
    tone: str = Field(description="Target tone, e.g. 'violet/pearl', 'ash', 'beige/gold', 'natural', 'copper'")
    technique: str = Field(description="e.g. 'global lightening + tone', 'balayage', 'root retouch', 'glaze/toning'")
    description: str = Field(description="Short professional description of the desired result, e.g. 'Clean, icy, level 10 platinum blonde. Cool, reflective finish. No yellow.'")

print('Desired state prompt and schema ready.')
print('Schema fields:', list(DesiredStateAnalysis.model_fields.keys()))

In [ ]:
def analyse_desired_state(client_notes: str, image_path: str = None) -> DesiredStateAnalysis:
    """Send client notes (and optionally a desired-look image) to Gemini and return structured analysis."""
    contents = [f"CLIENT SAYS: \"{client_notes}\""]
    
    if image_path:
        with open(image_path, 'rb') as f:
            image_bytes = f.read()
        contents.append(types.Part.from_bytes(data=image_bytes, mime_type='image/png'))
    
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config={
            'system_instruction': DESIRED_STATE_PROMPT,
            'response_mime_type': 'application/json',
            'response_schema': DesiredStateAnalysis,
            'temperature': 0.1,
        },
    )
    return DesiredStateAnalysis.model_validate_json(response.text)


def display_desired_state(title: str, result: DesiredStateAnalysis):
    """Pretty-print the desired state analysis."""
    print(f'\n{"+"*60}')
    print(f'  {title}')
    print(f'{"+"*60}\n')
    print(f'  Target Level:  {result.target_level}')
    print(f'  Tone:          {result.tone}')
    print(f'  Technique:     {result.technique}')
    print(f'  Description:   {result.description}')
    print()

print('Helper functions ready.')

### Case 1: Scandi Blonde — image + notes
Client says: *"I want to be super bright blonde all over, like an icy Scandi blonde. I hate seeing any yellow or gold."*

![Desired](../testcases/structured/case1_desired.png)

In [ ]:
result1 = analyse_desired_state(
    client_notes="I want to be super bright blonde all over, like an icy Scandi blonde. I hate seeing any yellow or gold.",
    image_path='../testcases/structured/case1_desired.png',
)
display_desired_state('Case 1: Scandi Blonde — Desired State', result1)

In [ ]:
# Load case 1 test data — we'll use this throughout the notebook
with open('../testcases/structured/case1_scandi_blonde.json') as f:
    case1 = json.load(f)

print(f"Title:       {case1['title']}")
print(f"Client says: {case1['client_description']}")
print(f"Start level: {case1['starting_point']['level']}")
print(f"Start desc:  {case1['starting_point']['description']}")
print(f"Colour theory: {case1['known_good_formulation']['colour_theory']}")

In [ ]:
# Known-good desired state — pulled from the test case JSON
# Note: the JSON doesn't have an explicit "desired state" object, so we show what's there
print('+' * 60)
print('  Case 1: KNOWN-GOOD (from Matt\'s test case JSON)')
print('+' * 60)
print()
print(f"  Title:         {case1['title']}")
print(f"  Client says:   {case1['client_description']}")
print(f"  Colour theory: {case1['known_good_formulation']['colour_theory']}")
print()
print('  (No explicit desired level/tone in JSON — the AI output below')
print('   should be consistent with the above.)')
print()

In [ ]:
# Step 1-2: Analyse the desired state — image + client notes
desired = analyse_desired_state(
    client_notes=case1['client_description'],
    image_path='../testcases/structured/case1_desired.png',
)
display_desired_state('Case 1: Scandi Blonde — AI Desired State', desired)

### Step 2: Confirm or correct

In the real app, the salonist sees the AI output above and either confirms or edits the fields before proceeding. For now we'll assume the AI got it right and carry the `desired` object forward.

---
## Step 3: Starting State — Photo + Consent

The salonist takes a photo of the client's current hair. We also ask for consent to save the photo for training.

Using `case1_start_v2.png` which shows the client's face (needed for preview generation later).

![Starting](../testcases/structured/case1_start_v2.png)

In [ ]:
STARTING_STATE_PROMPT = """\
You are an expert hair colour analyst. A salonist has taken a photo of their client's current hair.
They may also provide notes about the client's hair history.

Analyse the photo and return a structured assessment of the client's current hair state.
Be precise — this assessment will be used to plan a colour formulation.
"""

class StartingStateAnalysis(BaseModel):
    current_level: int = Field(description="Assessed natural/current level (1=black, 10=lightest blonde)")
    description: str = Field(description="Short professional description, e.g. 'Medium Blonde, virgin hair, healthy condition'")
    grey_percentage: int = Field(description="Estimated percentage of grey/white hair visible (0 if none)")
    condition: str = Field(description="Hair condition: healthy, slightly porous, damaged, dry, etc.")
    previous_colour: str = Field(description="Any visible signs of previous colour work, or 'none'")

print('Starting state prompt and schema ready.')
print('Schema fields:', list(StartingStateAnalysis.model_fields.keys()))

In [ ]:
def analyse_starting_state(image_path: str, stylist_notes: str = None) -> StartingStateAnalysis:
    """Send a client photo (and optional stylist notes) to Gemini and return structured assessment."""
    with open(image_path, 'rb') as f:
        image_bytes = f.read()
    
    contents = [types.Part.from_bytes(data=image_bytes, mime_type='image/png')]
    if stylist_notes:
        contents.append(f"STYLIST NOTES: \"{stylist_notes}\"")
    
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config={
            'system_instruction': STARTING_STATE_PROMPT,
            'response_mime_type': 'application/json',
            'response_schema': StartingStateAnalysis,
            'temperature': 0.1,
        },
    )
    return StartingStateAnalysis.model_validate_json(response.text)


def display_starting_state(title: str, result: StartingStateAnalysis):
    """Pretty-print the starting state analysis."""
    print(f'\n{"+"*60}')
    print(f'  {title}')
    print(f'{"+"*60}\n')
    print(f'  Current Level:    {result.current_level}')
    print(f'  Description:      {result.description}')
    print(f'  Grey %:           {result.grey_percentage}')
    print(f'  Condition:        {result.condition}')
    print(f'  Previous Colour:  {result.previous_colour}')
    print()

print('Starting state helpers ready.')

In [ ]:
# Step 3-4: AI analyses the starting state photo
starting = analyse_starting_state(
    image_path='../testcases/structured/case1_start_v2.png',
)
display_starting_state('Case 1: Scandi Blonde — AI Starting State', starting)

In [ ]:
# Known-good starting state — pulled from the test case JSON
sp = case1['starting_point']
print('+' * 60)
print('  Case 1: KNOWN-GOOD Starting State (from Matt\'s JSON)')
print('+' * 60)
print()
print(f"  Level:           {sp['level']}")
print(f"  Description:     {sp['description']}")
print(f"  Grey %:          {sp['grey_percentage']}")
print(f"  Condition:       {sp['condition']}")
print(f"  Previous colour: {sp['previous_colour']}")
print()

### Step 3 addendum: Consent

In the real app, this is where we ask: *"Does the client consent to their photo being saved for training?"* (yes/no toggle). If yes, save to a training dataset. No API call — pure UX.

### Step 4: Confirm or correct starting state

Same as step 2 — salonist reviews the AI output above and confirms or corrects. We carry the `starting` object forward.

---
## Step 5: Preview Generation (stretch)

Show the client a visual approximation of what they'd look like with the desired colour. This is the "wow moment" — but also the riskiest inference.

We send the **starting state photo** (with face) + the **desired state description** and ask for an edited image.

In [ ]:
# Step 5: Generate a preview image — client's face with desired hair colour
# Using gemini-2.5-flash-image which supports native image generation via generate_content

PREVIEW_PROMPT = f"""\
Edit this photo to change ONLY the hair colour. Keep the person's face, skin, clothing, 
and background exactly the same. 

Change the hair to: {desired.description}

Target level: {desired.target_level}
Tone: {desired.tone}

Make the result look natural and photorealistic. Do not change the hairstyle, length, or texture.
"""

with open('../testcases/structured/case1_start_v2.png', 'rb') as f:
    start_image_bytes = f.read()

preview_response = client.models.generate_content(
    model='gemini-2.5-flash-image',
    contents=[
        PREVIEW_PROMPT,
        types.Part.from_bytes(data=start_image_bytes, mime_type='image/png'),
    ],
    config=types.GenerateContentConfig(
        response_modalities=['IMAGE', 'TEXT'],
    ),
)

# Extract and display the generated image
from io import BytesIO
preview_image = None
for part in preview_response.candidates[0].content.parts:
    if part.inline_data and part.inline_data.mime_type.startswith('image/'):
        preview_image = Image.open(BytesIO(part.inline_data.data))
        display(preview_image)
        break

if not preview_image:
    print('No image generated. Model response:')
    print(preview_response.text)